Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет.
Здійснити data cleaning 

In [3]:
import pandas as pd
import numpy as np
import timeit


def load_and_clean_power_data(file_path):
    df = pd.read_csv(file_path, sep=';', 
                     low_memory=False, 
                     na_values=['?'])
    
    df = df.dropna()
    
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    df[numeric_cols] = df[numeric_cols].astype(float)

    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
    
    return df

df_power = load_and_clean_power_data('household_power_consumption.txt')

In [4]:
print(df_power.head())
print(f"Total rows: {len(df_power)}")

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
0              18.4             0.0             1.0            17.0   
1              23.0             0.0             1.0            16.0   
2              23.0             0.0             2.0            17.0   
3              23.0             0.0             1.0            17.0   
4              15.8             0.0             1.0            17.0   

             DateTime  
0 2006-12-16 17:24:00  
1 2006-12-16 17:25:00  
2 2006-12-16 17:26:0

Окремими функціями сформувати вибірки:
    1. Обрати всі записи, у яких загальна активна споживана 
    потужність перевищує 5 кВт.


In [6]:
def filter_high_power(df):
    return df[df['Global_active_power'] > 5]

Перевірка цього пункту:

In [7]:
filter_high_power(df_power).head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


2. Обрати всі записи, у яких сила струму лежить в межах 19
   20 А, для них виявити ті, у яких пральна машина та
   холодильних споживають більше, ніж бойлер та кондиціонер.

In [8]:
def filter_intensity_and_appliances(df):
    mask = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)
    result = df[mask & (df['Sub_metering_2'] > df['Sub_metering_3'])]
    return result

Перевірка для 2 пункту:

In [9]:
filter_intensity_and_appliances(df_power).head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


3. Обрати випадковим чином 500000 записів (без повторів
   елементів вибірки), для них обчислити середні величини
   усіх 3-х груп споживання електричної енергії


In [10]:
def random_sample_means(df):
    sample = df.sample(n=500000, replace=False)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means


Перевірка 3 пункту:

In [11]:
 random_sample_means(df_power).head()

Sub_metering_1    1.115306
Sub_metering_2    1.304930
Sub_metering_3    6.450716
dtype: float64

4. Обрати ті записи, які після 18-00 споживають понад 6 кВт
   за хвилину в середньому, серед відібраних визначити ті,
   у яких основне споживання електроенергії у вказаний
   проміжок часу припадає на пральну машину, сушарку,
   холодильник та освітлення (група 2 є найбільшою), а
   потім обрати кожен третій результат із першої половини
   та кожен четвертий результат із другої половини.


In [12]:
def complex_filter(df):
    time_mask = (df['DateTime'].dt.hour >= 18) & (df['Global_active_power'] > 6)
    initial_subset = df[time_mask]
    
    final_subset = initial_subset[
        (initial_subset['Sub_metering_2'] > initial_subset['Sub_metering_1']) & 
        (initial_subset['Sub_metering_2'] > initial_subset['Sub_metering_3'])
    ]
    
    mid = len(final_subset) // 2
    first_half = final_subset.iloc[:mid:3] 
    second_half = final_subset.iloc[mid::4]  
    
    return pd.concat([first_half, second_half])

Перевірка:

In [13]:
 complex_filter(df_power).head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


Пронормувати та стандартизувати вибраний датасет

In [14]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def scale_data(df):
    cols_to_scale = ['Global_active_power', 'Voltage']
    scaler_minmax = MinMaxScaler()
    df_minmax = pd.DataFrame(
        scaler_minmax.fit_transform(df[cols_to_scale]), 
        columns=[f'{col}_Norm' for col in cols_to_scale]
    )

    scaler_std = StandardScaler()
    df_std = pd.DataFrame(
        scaler_std.fit_transform(df[cols_to_scale]), 
        columns=[f'{col}_Std' for col in cols_to_scale]
    )
    comparison = pd.concat([df[cols_to_scale].head(), df_minmax.head(), df_std.head()], axis=1)
    return comparison

Перевірка нормалізації і стандартизації:

In [15]:
scaled_results = scale_data(df_power)
display(scaled_results)

,Global_active_power,Voltage,Global_active_power_Norm,Voltage_Norm,Global_active_power_Std,Voltage_Std
0,4.216,234.84,0.374796,0.376090,2.955077,-1.851816
1,5.360,233.63,0.478363,0.336995,4.037085,-2.225274
2,5.374,233.29,0.479631,0.326010,4.050326,-2.330213
3,5.388,233.74,0.480898,0.340549,4.063567,-2.191324
4,3.666,235.68,0.325005,0.403231,2.434881,-1.592556


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів

In [16]:
def calculate_correlations(df):
    attr1 = 'Global_active_power'
    attr2 = 'Global_intensity'
    pearson_corr = df[attr1].corr(df[attr2], method='pearson')
    spearman_corr = df[attr1].corr(df[attr2], method='spearman')
    
    print(f"Pearson corr: {pearson_corr:.4f}")
    print(f"Spearman corr: {spearman_corr:.4f}")
    
    return pearson_corr, spearman_corr

Перевірка коефіцієнтів Пірсона та Спірмена:

In [17]:
p_corr, s_corr = calculate_correlations(df_power)

Pearson corr: 0.9989
Spearman corr: 0.9954


Провести One Hot Encoding категоріального атрибута(Day_Type).

In [18]:
import numpy as np

def apply_one_hot_encoding(df):
    df['Day_Type'] = np.where(df['DateTime'].dt.dayofweek < 5, 'Weekday', 'Weekend')
    df_encoded = pd.get_dummies(df, columns=['Day_Type'])
    cols_to_show = ['DateTime', 'Day_Type_Weekday', 'Day_Type_Weekend']
    return df_encoded[cols_to_show].head(10)

Перевірка:

In [19]:
encoded_sample = apply_one_hot_encoding(df_power)
display(encoded_sample)

,DateTime,Day_Type_Weekday,Day_Type_Weekend
0,2006-12-16 17:24:00,False,True
1,2006-12-16 17:25:00,False,True
2,2006-12-16 17:26:00,False,True
3,2006-12-16 17:27:00,False,True
4,2006-12-16 17:28:00,False,True
5,2006-12-16 17:29:00,False,True
6,2006-12-16 17:30:00,False,True
7,2006-12-16 17:31:00,False,True
8,2006-12-16 17:32:00,False,True
9,2006-12-16 17:33:00,False,True


Аналіз часових витрат на виконання процедур  (профілювання часу виконання здійснено за допомогою модуля timeit).


In [20]:
def profile_and_report(task_name, task_func, df):
    start_time = timeit.default_timer()
    result = task_func(df)
    end_time = timeit.default_timer()
    
    execution_time = end_time - start_time
    print(f"{task_name:.<45} {execution_time:.6f} s")
    return execution_time

time1 = profile_and_report("Task 1 (Power > 5kW)", filter_high_power, df_power)
time2 = profile_and_report("Task 2 (Intensity 19-20A & Appliances)", filter_intensity_and_appliances, df_power)
time3 = profile_and_report("Task 3 (Random 500k Sample Means)", random_sample_means, df_power)
time4 = profile_and_report("Task 4 (Complex Filter & Step-indexing)", complex_filter, df_power)

print(f"Sum time: {time1 + time2 + time3 + time4:.4f} s")

Task 1 (Power > 5kW)......................... 0.008500 s
Task 2 (Intensity 19-20A & Appliances)....... 0.013800 s
Task 3 (Random 500k Sample Means)............ 0.284900 s
Task 4 (Complex Filter & Step-indexing)...... 0.041600 s
Sum time: 0.3488 s
